In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
df = spark.read.parquet('/Volumes/workspace/default/nyc_taxi')
df.count()
print((df.count(), len(df.columns)))
# this calculate the 5 number summary
summary = df.describe().toPandas()
summary

In [0]:
df.limit(10).toPandas()

In [0]:
from pyspark.sql.functions import col, sum
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas()
# df.duplicated() is not available for Spark DataFrames
# To check duplicates, use groupBy and count
un_cols=['VendorID','tpep_pickup_datetime','tpep_dropoff_datetime','PULocationID','DOLocationID']
uni_rows=df.groupBy(un_cols).count().filter((col("count") > 1) & (col("VendorID") != 2))
print(uni_rows.count())
display(uni_rows)
dup_keys = (
    df.groupBy(un_cols)
      .count()
      .filter(col("count") > 1)
)

display(dup_keys.limit(20))

    

In [0]:
duplicate_rows = (
    df.join(dup_keys.drop("count"), on=un_cols, how="inner")
      .orderBy(un_cols)
)

display(duplicate_rows.limit(100))

In [0]:
r_fare_rides = df.filter((col("fare_amount") > 0) & (col("tip_amount") > 0) & (col("total_amount") > 0)).count()
print(r_fare_rides)


In [0]:
negative_money = df.filter(
    (
    (col("fare_amount") < 0) |
    (col("tip_amount") < 0) |
    (col("total_amount") < 0)
    ) &
    (col("VendorID") != 2)
)
print(negative_money.count())
display(negative_money)

**Based on the current investigation, negative monetary values appear to be associated exclusively with VendorID = 2. These records are likely to represent financial reversal or correction transactions rather than ordinary taxi trips.**